In [1]:
%pip install seaborn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import models

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
CONFIG = {
    # Data
    'metadata_csv': 'processed_mels/all_metadata.csv',
    'target_frames': 256,
    
    # Training
    'batch_size': 32,
    'num_epochs': 50,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    
    # Optimization
    'patience_early_stop': 50,  # Set to 50 to disable early stopping (trains all 50 epochs)
    'patience_lr': 3,
    'lr_factor': 0.5,
    
    # Augmentation
    'use_augmentation': True,
    'aug_prob_time_mask': 0.5,
    'aug_prob_freq_mask': 0.5,
    'aug_prob_time_shift': 0.3,
    'aug_prob_noise': 0.3,
    
    # System
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'num_workers': 0,  # Changed from 4 - avoids Windows multiprocessing issues
    'random_seed': 42,
}

In [4]:
class MelSegmentDataset(Dataset):
    """
    Dataset for mel-spectrograms with optional augmentation.
    Includes SpecAugment-style masking and other audio augmentations.
    """
    def __init__(self, metadata_csv, target_frames=256, augment=False, 
                 aug_config=None):
        self.df = pd.read_csv(metadata_csv)
        self.paths = self.df['file'].tolist()
        self.labels = self.df['label'].tolist()
        
        # Create label mappings
        classes = sorted(set(self.labels))
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.idx_to_class = {i: c for c, i in self.class_to_idx.items()}
        self.y = [self.class_to_idx[l] for l in self.labels]
        
        self.target_frames = target_frames
        self.augment = augment
        self.aug_config = aug_config if aug_config else {}
        
    def _pad_or_crop(self, mel):
        """Pad or crop mel-spectrogram to target length"""
        n_mels, T = mel.shape
        if T == self.target_frames:
            return mel
        if T > self.target_frames:
            # Crop from center
            start = (T - self.target_frames) // 2
            end = start + self.target_frames
            return mel[:, start:end]
        # Pad
        pad_width = self.target_frames - T
        return np.pad(mel, ((0, 0), (0, pad_width)), mode='constant')
    
    def _augment_mel(self, mel):
        """
        Apply random augmentations to mel-spectrogram.
        Implements SpecAugment-style masking plus additional transforms.
        """
        # Time masking (mask horizontal stripes)
        if np.random.rand() < self.aug_config.get('prob_time_mask', 0.5):
            t_mask_size = np.random.randint(5, 30)
            t_start = np.random.randint(0, max(1, mel.shape[2] - t_mask_size))
            mel[:, :, t_start:t_start+t_mask_size] = 0
        
        # Frequency masking (mask vertical stripes)
        if np.random.rand() < self.aug_config.get('prob_freq_mask', 0.5):
            f_mask_size = np.random.randint(5, 20)
            f_start = np.random.randint(0, max(1, mel.shape[1] - f_mask_size))
            mel[:, f_start:f_start+f_mask_size, :] = 0
        
        # Time shifting (circular shift)
        if np.random.rand() < self.aug_config.get('prob_time_shift', 0.3):
            shift = np.random.randint(-20, 20)
            mel = torch.roll(mel, shifts=shift, dims=2)
        
        # Add Gaussian noise
        if np.random.rand() < self.aug_config.get('prob_noise', 0.3):
            noise = torch.randn_like(mel) * 0.01
            mel = torch.clamp(mel + noise, 0, 1)
        
        return mel
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        # Load mel-spectrogram
        mel = np.load(self.paths[idx])  # (n_mels, T)
        mel = self._pad_or_crop(mel)
        mel = mel[np.newaxis, :, :]     # (1, n_mels, T)
        mel = torch.from_numpy(mel).float()
        
        # Apply augmentation if training
        if self.augment:
            mel = self._augment_mel(mel)
        
        label = self.y[idx]
        return mel, label

In [5]:
def patient_independent_split(metadata_csv, train_ratio=0.7, val_ratio=0.15, 
                              random_seed=42):
    """
    Split dataset by patients, not segments.
    This prevents data leakage and ensures generalization.
    
    Args:
        metadata_csv: Path to metadata CSV
        train_ratio: Proportion for training
        val_ratio: Proportion for validation (rest goes to test)
        random_seed: Random seed for reproducibility
    
    Returns:
        train_idx, val_idx, test_idx: Lists of indices for each split
    """
    df = pd.read_csv(metadata_csv)
    
    # Extract patient ID from audio file path
    # Assumes structure: .../PatientXX/audio.wav
    df['patient_id'] = df['audio_file'].apply(lambda x: Path(x).parent.name)
    
    # Get unique patients
    patients = df['patient_id'].unique()
    n_patients = len(patients)
    n_train = int(n_patients * train_ratio)
    n_val = int(n_patients * val_ratio)
    
    # Shuffle patients with seed
    np.random.seed(random_seed)
    np.random.shuffle(patients)
    
    # Split patients
    train_patients = patients[:n_train]
    val_patients = patients[n_train:n_train+n_val]
    test_patients = patients[n_train+n_val:]
    
    # Get segment indices for each split
    train_idx = df[df['patient_id'].isin(train_patients)].index.tolist()
    val_idx = df[df['patient_id'].isin(val_patients)].index.tolist()
    test_idx = df[df['patient_id'].isin(test_patients)].index.tolist()
    
    print("=" * 60)
    print("PATIENT-INDEPENDENT SPLIT")
    print("=" * 60)
    print(f"Total patients: {n_patients}")
    print(f"  Train: {len(train_patients)} patients ({len(train_idx)} segments)")
    print(f"  Val:   {len(val_patients)} patients ({len(val_idx)} segments)")
    print(f"  Test:  {len(test_patients)} patients ({len(test_idx)} segments)")
    print("=" * 60)
    
    return train_idx, val_idx, test_idx

In [6]:
def create_resnet_model(num_classes, model_name='resnet18', pretrained=True):
    """
    Create ResNet model adapted for mel-spectrograms.
    
    Args:
        num_classes: Number of output classes
        model_name: 'resnet18', 'resnet34', or 'resnet50'
        pretrained: Whether to use ImageNet pretrained weights
    
    Returns:
        model: Modified ResNet model
    """
    # Load base model
    if model_name == 'resnet18':
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        model = models.resnet18(weights=weights)
    elif model_name == 'resnet34':
        weights = models.ResNet34_Weights.DEFAULT if pretrained else None
        model = models.resnet34(weights=weights)
    elif model_name == 'resnet50':
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        model = models.resnet50(weights=weights)
    else:
        raise ValueError(f"Unknown model: {model_name}")
    
    # Modify first conv layer: 3 channels (RGB) -> 1 channel (mel-spectrogram)
    old_conv = model.conv1
    model.conv1 = nn.Conv2d(
        in_channels=1,
        out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=old_conv.bias is not None
    )
    
    # If using pretrained weights, initialize new conv1 by averaging RGB channels
    if pretrained:
        with torch.no_grad():
            model.conv1.weight[:] = old_conv.weight.mean(dim=1, keepdim=True)
    
    # Modify final FC layer for number of classes
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    
    print(f"Created {model_name} with {num_classes} output classes")
    print(f"Pretrained: {pretrained}")
    
    return model


In [7]:
def get_class_weights(dataset, indices):
    """
    Compute class weights for handling class imbalance.
    Uses 'balanced' strategy from sklearn.
    
    Args:
        dataset: MelSegmentDataset instance
        indices: List of indices for the training set
    
    Returns:
        class_weights: Tensor of weights for each class
    """
    # Get labels for training set
    train_labels = [dataset.y[i] for i in indices]
    classes = np.unique(train_labels)
    
    # Compute balanced weights
    weights = compute_class_weight(
        'balanced',
        classes=classes,
        y=train_labels
    )
    
    class_weights = torch.FloatTensor(weights)
    
    print("\nClass Weights (for handling imbalance):")
    for i, w in enumerate(class_weights):
        class_name = dataset.idx_to_class[i]
        count = train_labels.count(i)
        print(f"  {class_name}: {w:.3f} (n={count})")
    
    return class_weights

In [8]:
class Trainer:
    """
    Trainer class with all optimizations:
    - Weighted loss for class imbalance
    - Learning rate scheduling
    - Early stopping
    - Model checkpointing
    - F1-score tracking
    """
    def __init__(self, model, train_loader, val_loader, config, class_weights=None):
        self.model = model.to(config['device'])
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.config = config
        self.device = config['device']
        
        # Loss function with class weights
        if class_weights is not None:
            class_weights = class_weights.to(self.device)
        self.criterion = nn.CrossEntropyLoss(weight=class_weights)
        
        # Optimizer
        self.optimizer = torch.optim.Adam(
            model.parameters(),
            lr=config['lr'],
            weight_decay=config['weight_decay']
        )
        
        # Learning rate scheduler
        self.scheduler = ReduceLROnPlateau(
            self.optimizer,
            mode='max',  # Maximize val F1-score
            factor=config['lr_factor'],
            patience=config['patience_lr']
        )
        
        # Early stopping
        self.best_f1 = 0.0
        self.early_stop_counter = 0
        self.patience_early_stop = config['patience_early_stop']
        
        # History
        self.history = {
            'train_loss': [],
            'train_acc': [],
            'val_acc': [],
            'val_f1': [],
            'lr': []
        }
    
    def train_epoch(self):
        """Train for one epoch"""
        self.model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        # Add progress bar
        from tqdm import tqdm
        pbar = tqdm(self.train_loader, desc='Training', leave=False)
        
        for batch_idx, (X, y) in enumerate(pbar):
            X, y = X.to(self.device), y.to(self.device)
            
            # Forward pass
            self.optimizer.zero_grad()
            outputs = self.model(X)
            loss = self.criterion(outputs, y)
            
            # Backward pass
            loss.backward()
            self.optimizer.step()
            
            # Track metrics
            total_loss += loss.item() * X.size(0)
            _, preds = outputs.max(1)
            correct += (preds == y).sum().item()
            total += y.size(0)
            
            # Update progress bar every 10 batches
            if batch_idx % 10 == 0:
                current_loss = total_loss / max(1, total)
                current_acc = correct / max(1, total)
                pbar.set_postfix({
                    'loss': f'{current_loss:.4f}',
                    'acc': f'{current_acc:.3f}'
                })
        
        pbar.close()
        avg_loss = total_loss / total
        accuracy = correct / total
        return avg_loss, accuracy
    
    @torch.no_grad()
    def validate(self):
        """Validate on validation set"""
        self.model.eval()
        all_preds = []
        all_true = []
        
        for X, y in self.val_loader:
            X, y = X.to(self.device), y.to(self.device)
            outputs = self.model(X)
            _, preds = outputs.max(1)
            
            all_preds.extend(preds.cpu().numpy())
            all_true.extend(y.cpu().numpy())
        
        # Compute metrics
        accuracy = np.mean(np.array(all_preds) == np.array(all_true))
        f1 = f1_score(all_true, all_preds, average='macro')
        
        return accuracy, f1, all_true, all_preds
    
    def train(self):
        """Main training loop"""
        print("\n" + "=" * 60)
        print("STARTING TRAINING")
        print("=" * 60)
        
        # Display GPU memory info if using CUDA
        if torch.cuda.is_available():
            print(f"GPU Memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
            print(f"GPU Memory reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
            print("=" * 60)
        
        for epoch in range(self.config['num_epochs']):
            # Train
            train_loss, train_acc = self.train_epoch()
            
            # Validate
            val_acc, val_f1, _, _ = self.validate()
            
            # Update learning rate based on validation F1
            self.scheduler.step(val_f1)
            current_lr = self.optimizer.param_groups[0]['lr']
            
            # Save history
            self.history['train_loss'].append(train_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_acc'].append(val_acc)
            self.history['val_f1'].append(val_f1)
            self.history['lr'].append(current_lr)
            
            # Print progress
            print(f"\nEpoch {epoch+1}/{self.config['num_epochs']}")
            print(f"  Train - Loss: {train_loss:.4f}, Acc: {train_acc:.3f}")
            print(f"  Val   - Acc: {val_acc:.3f}, F1: {val_f1:.3f}")
            print(f"  LR: {current_lr:.6f}")
            
            # Save best model
            if val_f1 > self.best_f1:
                self.best_f1 = val_f1
                self.early_stop_counter = 0
                self.save_checkpoint('best_model.pth', epoch, val_f1)
                print(f"  ✓ New best model saved (F1: {val_f1:.3f})")
            else:
                self.early_stop_counter += 1
            
            # Early stopping
            if self.early_stop_counter >= self.patience_early_stop:
                print(f"\n Early stopping triggered at epoch {epoch+1}")
                print(f"  No improvement for {self.patience_early_stop} epochs")
                break
            
            # Clear GPU cache after each epoch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        print("\n" + "=" * 60)
        print(f"TRAINING COMPLETE - Best F1: {self.best_f1:.3f}")
        print("=" * 60)
    
    def save_checkpoint(self, path, epoch, val_f1):
        """Save model checkpoint"""
        # Create directory if it doesn't exist
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'val_f1': val_f1,
            'config': self.config,
            'history': self.history,  # Save training history
        }, path)
    
    def load_checkpoint(self, path):
        """Load model checkpoint"""
        checkpoint = torch.load(path)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        return checkpoint

In [9]:
@torch.no_grad()
def evaluate_model(model, test_loader, device, class_names, save_path='results'):
    """
    Comprehensive model evaluation with multiple metrics.
    
    Args:
        model: Trained model
        test_loader: DataLoader for test set
        device: Device to run on
        class_names: List of class names
        save_path: Directory to save results
    
    Returns:
        results: Dictionary with all metrics
    """
    # Create results directory if it doesn't exist
    Path(save_path).mkdir(parents=True, exist_ok=True)
    
    model.eval()
    all_preds = []
    all_true = []
    all_probs = []
    
    # Collect predictions
    for X, y in test_loader:
        X = X.to(device)
        outputs = model(X)
        probs = F.softmax(outputs, dim=1)
        _, preds = outputs.max(1)
        
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(y.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_true = np.array(all_true)
    all_probs = np.array(all_probs)
    
    # Create save directory
    Path(save_path).mkdir(exist_ok=True)
    
    # ========== Classification Report ==========
    print("\n" + "=" * 60)
    print("CLASSIFICATION REPORT")
    print("=" * 60)
    report = classification_report(all_true, all_preds, 
                                  target_names=class_names,
                                  digits=3)
    print(report)
    
    # Save report
    with open(f'{save_path}/classification_report.txt', 'w') as f:
        f.write(report)
    
    # ========== Confusion Matrix ==========
    cm = confusion_matrix(all_true, all_preds)
    
    # Plot confusion matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names,
                yticklabels=class_names,
                cbar_kws={'label': 'Count'})
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{save_path}/confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"\nConfusion matrix saved to {save_path}/confusion_matrix.png")
    
    # ========== Per-Class Metrics ==========
    from sklearn.metrics import precision_recall_fscore_support
    
    precision, recall, f1, support = precision_recall_fscore_support(
        all_true, all_preds, average=None
    )
    
    print("\n" + "=" * 60)
    print("PER-CLASS METRICS")
    print("=" * 60)
    print(f"{'Class':<15} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
    print("-" * 60)
    for i, name in enumerate(class_names):
        print(f"{name:<15} {precision[i]:<12.3f} {recall[i]:<12.3f} "
              f"{f1[i]:<12.3f} {support[i]:<10}")
    
    # ========== Overall Metrics ==========
    accuracy = np.mean(all_preds == all_true)
    macro_f1 = f1_score(all_true, all_preds, average='macro')
    weighted_f1 = f1_score(all_true, all_preds, average='weighted')
    
    print("\n" + "=" * 60)
    print("OVERALL METRICS")
    print("=" * 60)
    print(f"Accuracy:     {accuracy:.3f}")
    print(f"Macro F1:     {macro_f1:.3f}")
    print(f"Weighted F1:  {weighted_f1:.3f}")
    print("=" * 60)
    
    # Compile results
    results = {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'per_class_precision': precision,
        'per_class_recall': recall,
        'per_class_f1': f1,
        'confusion_matrix': cm,
        'predictions': all_preds,
        'true_labels': all_true,
        'probabilities': all_probs,
    }
    
    # Save results
    np.save(f'{save_path}/results.npy', results)
    
    return results


def plot_training_history(history, save_path='results'):
    """Plot training history"""
    # Create results directory if it doesn't exist
    Path(save_path).mkdir(parents=True, exist_ok=True)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Loss
    axes[0, 0].plot(history['train_loss'], label='Train Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[0, 1].plot(history['train_acc'], label='Train Acc')
    axes[0, 1].plot(history['val_acc'], label='Val Acc')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].set_title('Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # F1-Score
    axes[1, 0].plot(history['val_f1'], label='Val F1', color='green')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('F1-Score')
    axes[1, 0].set_title('Validation F1-Score')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Learning Rate
    axes[1, 1].plot(history['lr'], label='Learning Rate', color='orange')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Learning Rate')
    axes[1, 1].set_title('Learning Rate Schedule')
    axes[1, 1].set_yscale('log')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{save_path}/training_history.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Training history saved to {save_path}/training_history.png")

In [10]:
def main():
    """Main execution function"""
    # Set random seeds for reproducibility
    torch.manual_seed(CONFIG['random_seed'])
    np.random.seed(CONFIG['random_seed'])
    
    print("=" * 60)
    print("OPTIMIZED RESNET FOR OSA DETECTION")
    print("=" * 60)
    print(f"Device: {CONFIG['device']}")
    print(f"Random seed: {CONFIG['random_seed']}")
    
    # ========== Pre-flight Checks ==========
    print("\n" + "=" * 60)
    print("PRE-FLIGHT CHECKS")
    print("=" * 60)
    
    # Check 1: Verify metadata CSV exists
    metadata_path = Path(CONFIG['metadata_csv'])
    if not metadata_path.exists():
        print(f" ERROR: Metadata file not found!")
        print(f"   Expected: {metadata_path}")
        print("\nSolution:")
        print("1. Run: python preprocessing_gpu_accelerated.py")
        print("2. Run: python generate_normal_segments.py")
        print("3. Then re-run this training script")
        raise FileNotFoundError(f"Metadata file not found: {metadata_path}")
    print(f" Metadata file found: {metadata_path}")
    
    # Check 2: Verify CSV has both classes
    df_check = pd.read_csv(CONFIG['metadata_csv'])
    unique_labels = df_check['label'].unique()
    print(f" Unique labels in dataset: {sorted(unique_labels)}")
    
    if len(unique_labels) < 2:
        print(f"\n ERROR: Need at least 2 classes, found {len(unique_labels)}")
        print(f"   Current classes: {sorted(unique_labels)}")
        print("\nThis means you haven't generated normal segments yet!")
        print("\nSolution:")
        print("1. Run: python generate_normal_segments.py")
        print("2. Verify all_metadata.csv has both 'apnea' and 'normal' labels")
        print("3. Then re-run this training script")
        raise ValueError(f"Need at least 2 classes for classification")
    
    print(f" Found {len(unique_labels)} classes: {sorted(unique_labels)}")
    
    # Check 3: Verify CUDA availability
    if torch.cuda.is_available():
        print(f" CUDA available: {torch.cuda.get_device_name(0)}")
        print(f"   CUDA version: {torch.version.cuda}")
    else:
        print("  WARNING: CUDA not available, using CPU")
        print("   Training will be significantly slower on CPU")
    
    print("=" * 60)
    print(" All pre-flight checks passed!")
    print("=" * 60)
    
    # ========== Data Loading ==========
    print("\n[1/6] Loading data...")
    
    # Patient-independent split
    train_idx, val_idx, test_idx = patient_independent_split(
        CONFIG['metadata_csv'],
        random_seed=CONFIG['random_seed']
    )
    
    # Load metadata to check class distribution in splits
    df = pd.read_csv(CONFIG['metadata_csv'])
    
    print("\n" + "=" * 60)
    print("CLASS DISTRIBUTION IN SPLITS")
    print("=" * 60)
    
    # Check train split
    train_labels = df.iloc[train_idx]['label'].value_counts()
    print("Training set:")
    for label, count in train_labels.items():
        print(f"  {label}: {count}")
    
    # Check val split
    val_labels = df.iloc[val_idx]['label'].value_counts()
    print("\nValidation set:")
    for label, count in val_labels.items():
        print(f"  {label}: {count}")
    
    # Check test split
    test_labels = df.iloc[test_idx]['label'].value_counts()
    print("\nTest set:")
    for label, count in test_labels.items():
        print(f"  {label}: {count}")
    
    # Validate that train set has both classes
    if len(train_labels) < 2:
        print("\n" + "=" * 60)
        print("  WARNING: Training set only has 1 class!")
        print("=" * 60)
        print(f"Classes in training set: {train_labels.index.tolist()}")
        print("\nThis will cause training to fail!")
        print("Check that your all_metadata.csv has both 'apnea' and 'normal' classes")
        print("=" * 60)
    
    print("=" * 60)
    
    # Create datasets
    aug_config = {
        'prob_time_mask': CONFIG['aug_prob_time_mask'],
        'prob_freq_mask': CONFIG['aug_prob_freq_mask'],
        'prob_time_shift': CONFIG['aug_prob_time_shift'],
        'prob_noise': CONFIG['aug_prob_noise'],
    }
    
    full_dataset = MelSegmentDataset(
        CONFIG['metadata_csv'],
        target_frames=CONFIG['target_frames'],
        augment=False
    )
    
    train_dataset = MelSegmentDataset(
        CONFIG['metadata_csv'],
        target_frames=CONFIG['target_frames'],
        augment=CONFIG['use_augmentation'],
        aug_config=aug_config
    )
    
    train_ds = Subset(train_dataset, train_idx)
    val_ds = Subset(full_dataset, val_idx)
    test_ds = Subset(full_dataset, test_idx)
    
    # Create dataloaders
    use_cuda = torch.cuda.is_available()
    train_loader = DataLoader(
        train_ds,
        batch_size=CONFIG['batch_size'],
        shuffle=True,
        num_workers=CONFIG['num_workers'],
        pin_memory=use_cuda,  # Only use pin_memory with CUDA
        persistent_workers=True if CONFIG['num_workers'] > 0 else False
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=CONFIG['batch_size'],
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=use_cuda,
        persistent_workers=True if CONFIG['num_workers'] > 0 else False
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=CONFIG['batch_size'],
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=use_cuda,
        persistent_workers=True if CONFIG['num_workers'] > 0 else False
    )
    
    print(f"Augmentation: {CONFIG['use_augmentation']}")
    
    # ========== Validate Dataset ==========
    print("\n" + "=" * 60)
    print("DATASET VALIDATION")
    print("=" * 60)
    print(f"Total segments: {len(full_dataset)}")
    print(f"Classes detected: {list(full_dataset.class_to_idx.keys())}")
    print(f"Number of classes: {len(full_dataset.class_to_idx)}")
    
    # Check class distribution
    from collections import Counter
    label_counts = Counter(full_dataset.labels)
    print("\nClass distribution:")
    for label, count in label_counts.items():
        print(f"  {label}: {count}")
    
    # CRITICAL: Validate we have both classes
    if len(full_dataset.class_to_idx) < 2:
        print("\n" + "=" * 60)
        print(" ERROR: Dataset must have at least 2 classes!")
        print("=" * 60)
        print("\nDetected classes:", list(full_dataset.class_to_idx.keys()))
        print("\nPossible causes:")
        print("1. You haven't run 'generate_normal_segments.py' yet")
        print("2. The all_metadata.csv only contains apnea segments")
        print("3. The CSV file is outdated")
        print("\nSolution:")
        print("1. Run: python generate_normal_segments.py")
        print("2. Verify all_metadata.csv has both 'apnea' and 'normal' labels")
        print("3. Then re-run this training script")
        print("=" * 60)
        raise ValueError(f"Need at least 2 classes for classification, got {len(full_dataset.class_to_idx)}")
    
    print(" Dataset validation passed!")
    print("=" * 60)
    
    # ========== Model Creation ==========
    print("\n[2/6] Creating model...")
    num_classes = len(full_dataset.class_to_idx)
    model = create_resnet_model(num_classes, model_name='resnet18', pretrained=True)
    
    # Verify CUDA usage
    print("\n" + "=" * 60)
    print("DEVICE INFORMATION")
    print("=" * 60)
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA version: {torch.version.cuda}")
        print(f"Device name: {torch.cuda.get_device_name(0)}")
        print(f"Device count: {torch.cuda.device_count()}")
        print(f"Current device: {torch.cuda.current_device()}")
        print(f"Model on CUDA: {next(model.parameters()).is_cuda}")
    print("=" * 60)
    
    # ========== Class Weights ==========
    print("\n[3/6] Computing class weights...")
    class_weights = get_class_weights(full_dataset, train_idx)
    
    # ========== Training ==========
    print("\n[4/6] Training model...")
    trainer = Trainer(model, train_loader, val_loader, CONFIG, class_weights)
    trainer.train()
    
    # Plot training history
    plot_training_history(trainer.history)
    
    # ========== Load Best Model ==========
    print("\n[5/6] Loading best model...")
    checkpoint = trainer.load_checkpoint('best_model.pth')
    print(f"Loaded model from epoch {checkpoint['epoch']+1}")
    print(f"Best validation F1: {checkpoint['val_f1']:.3f}")
    
    # ========== Evaluation ==========
    print("\n[6/6] Evaluating on test set...")
    class_names = [full_dataset.idx_to_class[i] for i in range(num_classes)]
    results = evaluate_model(model, test_loader, CONFIG['device'], class_names)
    
    print("\n" + "=" * 60)
    print("DONE!")
    print("=" * 60)
    print("Results saved to 'results/' directory")


if __name__ == "__main__":
    main()

OPTIMIZED RESNET FOR OSA DETECTION
Device: cuda
Random seed: 42

PRE-FLIGHT CHECKS
 Metadata file found: processed_mels\all_metadata.csv
 Unique labels in dataset: ['apnea', 'normal']
 Found 2 classes: ['apnea', 'normal']
 CUDA available: NVIDIA GeForce GTX 1650
   CUDA version: 11.8
 All pre-flight checks passed!

[1/6] Loading data...
PATIENT-INDEPENDENT SPLIT
Total patients: 50
  Train: 35 patients (34066 segments)
  Val:   7 patients (4105 segments)
  Test:  8 patients (6673 segments)

CLASS DISTRIBUTION IN SPLITS
Training set:
  apnea: 18507
  normal: 15559

Validation set:
  apnea: 2178
  normal: 1927

Test set:
  apnea: 3570
  normal: 3103
Augmentation: True

DATASET VALIDATION
Total segments: 44844
Classes detected: ['apnea', 'normal']
Number of classes: 2

Class distribution:
  apnea: 24255
  normal: 20589
 Dataset validation passed!

[2/6] Creating model...
Created resnet18 with 2 output classes
Pretrained: True

DEVICE INFORMATION
PyTorch version: 2.7.1+cu118
CUDA available:


Epoch 1/50
  Train - Loss: 0.4902, Acc: 0.726
  Val   - Acc: 0.760, F1: 0.760
  LR: 0.000100
  ✓ New best model saved (F1: 0.760)



Epoch 2/50
  Train - Loss: 0.4632, Acc: 0.746
  Val   - Acc: 0.731, F1: 0.725
  LR: 0.000100



Epoch 3/50
  Train - Loss: 0.4535, Acc: 0.755
  Val   - Acc: 0.718, F1: 0.707
  LR: 0.000100



Epoch 4/50
  Train - Loss: 0.4401, Acc: 0.770
  Val   - Acc: 0.736, F1: 0.734
  LR: 0.000100


...


Epoch 47/50
  Train - Loss: 0.1514, Acc: 0.936
  Val   - Acc: 0.708, F1: 0.708
  LR: 0.000000



Epoch 48/50
  Train - Loss: 0.1532, Acc: 0.936
  Val   - Acc: 0.707, F1: 0.707
  LR: 0.000000



Epoch 49/50
  Train - Loss: 0.1482, Acc: 0.939
  Val   - Acc: 0.713, F1: 0.713
  LR: 0.000000



Epoch 50/50
  Train - Loss: 0.1496, Acc: 0.937
  Val   - Acc: 0.705, F1: 0.705
  LR: 0.000000

TRAINING COMPLETE - Best F1: 0.760
Training history saved to results/training_history.png

[5/6] Loading best model...


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn 